In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%pip install -U torch torchvision transformers datasets evaluate accelerate scikit-learn emoji peft

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

import evaluate
import numpy as np
from transformers import DataCollatorWithPadding

import re

In [ ]:
temp = load_dataset("csv", data_files="/content/drive/MyDrive/Copy of Sentiment140.csv", encoding="ISO-8859-1")["train"]
temp = temp.remove_columns(["ID", "Date", "Flag", "User"])
temp = temp.filter(lambda x: x["Label"] in [0, 4])

def relabel(batch):
    labels = batch["Label"]
    new_labels = [0 if l == 0 else 1 if l == 4 else None for l in labels]
    if any(l is None for l in new_labels):
        raise ValueError("Unexpected label in batch")
    return {"Label": new_labels}

def replace(text):
    text = re.sub(r'@\w+', '@USER', text)
    text = re.sub(r'http\S+|www\.\S+', 'HTTPURL', text)
    return text

def clean_text(batch):
    return {"Text": [replace(t) for t in batch["Text"]] }

temp = temp.map(relabel, batched=True)
temp = temp.map(clean_text, batched=True)
temp = temp.train_test_split(test_size=0.1, seed=1)
test_valid = temp["test"].train_test_split(test_size=0.5, seed=1)

dataset_dict = {
    "train": temp["train"].shuffle(seed=1),
    "test": test_valid["test"],
    "validation": test_valid["train"],
}

dataset_dict

In [ ]:
model_path = "vinai/bertweet-base"
tokenizer = AutoTokenizer.from_pretrained(model_path)

id2label = {0: "negative", 1: "positive"}
label2id = {"negative": 0, "positive": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    model_path, num_labels=2, id2label=id2label, label2id=label2id)

In [ ]:
max_positions = model.config.max_position_embeddings
num_virtual_tokens = 20
max_input_len = max_positions - 2 - num_virtual_tokens
if max_input_len <= 0:
    raise ValueError(f"max_input_len={max_input_len} is invalid; reduce num_virtual_tokens")

def preprocess_function(batch):
    return tokenizer(batch["Text"], truncation=True, padding=False, max_length=max_input_len)

tokenized = {}
for split, ds in dataset_dict.items():
    ds = ds.rename_column("Label", "labels")
    tokenized[split] = ds.map(preprocess_function, batched=True, remove_columns=["Text"])

In [ ]:
from peft import PromptTuningConfig, PromptTuningInit, TaskType

NUM_VIRTUAL_TOKENS = 20

prompt_config = PromptTuningConfig(
    task_type=TaskType.SEQ_CLS,
    num_virtual_tokens=NUM_VIRTUAL_TOKENS,
    prompt_tuning_init=PromptTuningInit.TEXT,
    prompt_tuning_init_text="Classify the sentiment of this tweet as positive or negative:",
    tokenizer_name_or_path=model_path,
 )

In [ ]:
from peft import get_peft_model

model = get_peft_model(model, prompt_config)

base = model.get_base_model() if hasattr(model, "get_base_model") else None
if base is None:
    base = getattr(model, "base_model", model)

unfrozen = []
for attr in ("classifier", "score"):
    head = getattr(base, attr, None)
    if head is not None and hasattr(head, "parameters"):
        for p in head.parameters():
            p.requires_grad = True
        unfrozen.append(attr)

print("Unfroze head modules:", unfrozen if unfrozen else "<not found>")
model.print_trainable_parameters()

In [ ]:
import os

RESUME_CHECKPOINT = "/content/drive/MyDrive/outputs/prompt-bertweet/checkpoint-45000"

#resume_from_checkpoint = None
resume_from_checkpoint = RESUME_CHECKPOINT if (RESUME_CHECKPOINT and os.path.isdir(RESUME_CHECKPOINT)) else None

if resume_from_checkpoint:
    print(f"Resuming from checkpoint: {resume_from_checkpoint}")
else:
    print("No checkpoint found (or not set). Training from scratch.")

In [ ]:
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer
from transformers.utils.notebook import NotebookProgressCallback
import evaluate, numpy as np

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/outputs/prompt-bertweet",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=5e-4,
    logging_steps=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=3,
    report_to="none",   # Colab Error
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Colab Error
trainer.remove_callback(NotebookProgressCallback)

trainer.train(resume_from_checkpoint=resume_from_checkpoint)
results = trainer.evaluate(eval_dataset=tokenized["test"])
print("Test accuracy:", results["eval_accuracy"])
print("Test F1:", results["eval_f1"])

In [ ]:
model.save_pretrained('/content/drive/MyDrive/outputs/prompt-bertweet/final')
tokenizer.save_pretrained('/content/drive/MyDrive/outputs/prompt-bertweet/final')

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
from transformers.utils.notebook import NotebookProgressCallback
from peft import PeftConfig, PeftModel
import evaluate
import numpy as np

model_dir = "/content/drive/MyDrive/outputs/prompt-bertweet/final"

peft_config = PeftConfig.from_pretrained(model_dir)

tokenizer = AutoTokenizer.from_pretrained(model_dir)

base_model = AutoModelForSequenceClassification.from_pretrained(
    peft_config.base_model_name_or_path,
    num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1},
)

model = PeftModel.from_pretrained(base_model, model_dir)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

eval_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/outputs/eval-prompt-bertweet",
    per_device_eval_batch_size=64,
    report_to="none",   # Colab Error
    do_train=False,
    do_eval=True,
)

trainer = Trainer(
    model=model,
    args=eval_args,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Colab Error
trainer.remove_callback(NotebookProgressCallback)

results = trainer.evaluate(eval_dataset=tokenized["test"])
print("Test accuracy:", results.get("eval_accuracy"))
print("Test F1:", results.get("eval_f1"))